# Fine-tunning and testing YOLOv11 Nano

In this notebook, we will explore the ultralytics package to fine-tune and test a YoLo ([You Only Look Once](https://docs.ultralytics.com)) Model. In this case, we will be using the nano version (smaller model), which will alleviate the requirement for GPU memory and facilitate our fine-tuning. Moreover, the model will also enable faster inference.

The cell below installs the necessary package (comment on it if you already have installed the package)

In [1]:
# Uncomment when running outside the provided Docker environment:
# %pip install ultralytics onnx onnxscript onnxslim

In [2]:
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO

# Locate the dataset whether Jupyter starts in the repository root or notebooks/.
dataset_candidates = [
    Path.cwd() / "images/object-detection/parasites",
    Path.cwd().parent / "images/object-detection/parasites",
]
DATASET_ROOT = next((path.resolve() for path in dataset_candidates if path.is_dir()), None)
if DATASET_ROOT is None:
    raise FileNotFoundError(
        "Could not find images/object-detection/parasites from the current directory "
        f"({Path.cwd()})."
    )

# Ultralytics requires a dataset YAML. Generate one with an absolute root so all
# split paths resolve consistently inside or outside the Docker container.
dataset_config = Path.cwd() / "parasites.yaml"
dataset_config.write_text(
    yaml.safe_dump(
        {
            "path": str(DATASET_ROOT),
            "train": "images/train",
            "val": "images/val",
            "test": "images/test",
            "names": {0: "parasite_egg"},
        },
        sort_keys=False,
    ),
    encoding="utf-8",
)
print(f"Dataset config: {dataset_config}")
print(f"Dataset root: {DATASET_ROOT}")

Dataset config: /workspace/notebooks/parasites.yaml
Dataset root: /workspace/images/object-detection/parasites


We start by instantiating our model.

In [3]:
model = YOLO("yolo11n.pt")

Then, we train it using our pre-trained model. The setup cell creates the required `parasites.yaml` with paths to the train, validation, and test splits. Training options can still be overwritten here (for instance, setting epochs to 200).

In [4]:
results = model.train(
    data=str(dataset_config),
    epochs=10,
    imgsz=640,
    plots=True,
    save=True,
    device=0  # Use GPU. Set to 'cpu' if no GPU is available
)

New https://pypi.org/project/ultralytics/8.4.103 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.96 🚀 Python-3.12.8 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine/trainer: task=detect, mode=train, model=yolo11n.pt, data=/workspace/notebooks/parasites.yaml, epochs=10, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=None, name=train4, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None

train: Scanning /workspace/images/object-detection/parasites/labels/train.cache... 379 images, 352 backgrounds, 0 corrupt: 100%|██████████| 731/731 [00:00<?, ?it/s]
val: Scanning /workspace/images/object-detection/parasites/labels/val.cache... 126 images, 117 backgrounds, 0 corrupt: 100%|██████████| 243/243 [00:00<?, ?it/s]


Plotting labels to runs/detect/train4/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to runs/detect/train4
Starting training for 10 epochs...
Closing dataloader mosaic

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/10      2.35G      1.309      4.536      1.311          4        640: 100%|██████████| 46/46 [00:15<00:00,  2.99it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.33it/s]

                   all        243        126          1     0.0421      0.545      0.324


...

      10/10       2.7G     0.9279     0.8663      1.009          6        640: 100%|██████████| 46/46 [00:05<00:00,  8.93it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:01<00:00,  7.38it/s]

                   all        243        126       0.93      0.956       0.98      0.733



10 epochs completed in 0.023 hours.
Optimizer stripped from runs/detect/train4/weights/last.pt, 5.5MB
Optimizer stripped from runs/detect/train4/weights/best.pt, 5.5MB

Validating runs/detect/train4/weights/best.pt...
Ultralytics 8.3.96 🚀 Python-3.12.8 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:01<00:00,  5.68it/s]


                   all        243        126       0.93      0.955       0.98      0.733
Speed: 0.2ms preprocess, 1.2ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to runs/detect/train4


Once the model is trained, we can run it on the validation subset to get metrics (you need to properly configure the input folders).

In [5]:
# Validate the model on the validation set
val_results = model.val()

Ultralytics 8.3.96 🚀 Python-3.12.8 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs


val: Scanning /workspace/images/object-detection/parasites/labels/val.cache... 126 images, 117 backgrounds, 0 corrupt: 100%|██████████| 243/243 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 16/16 [00:02<00:00,  6.28it/s]


                   all        243        126       0.93      0.955      0.981      0.732
Speed: 0.8ms preprocess, 3.3ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to runs/detect/train42


We can also run inference on a single image (the method will show where the output image was saved):

In [6]:
# Perform inference on a test image
test_image = DATASET_ROOT / "images/test/000002.png"
results = model.predict(str(test_image), save=True, conf=0.25)


image 1/1 /workspace/images/object-detection/parasites/images/test/000002.png: 640x640 1 parasite_egg, 8.2ms
Speed: 1.7ms preprocess, 8.2ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 640)
Results saved to runs/detect/train43


Another interesting feature of this approach is the capability to export the model to the onyx format, which enables a myriad of features, such as Runtime Optimization, Hardware Acceleration, and model combability across multiple types of devices (e.g., embedded systems, mobile, server).

In [7]:
# Export the model to ONNX format for deployment
# PyTorch's ONNX exporter requires both `onnx` and `onnxscript`.
try:
    import onnx
    import onnxscript
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Missing ONNX export dependencies. Run `%pip install onnx onnxscript`, "
        "restart the kernel, then rerun this cell."
    ) from exc

onnx_path = model.export(format="onnx", opset=18)
onnx.checker.check_model(onnx.load(onnx_path))
print(f"Validated ONNX model: {onnx_path}")

Ultralytics 8.3.96 🚀 Python-3.12.8 torch-2.10.0+cu128 CPU (AMD Ryzen 5 7600X 6-Core Processor)

PyTorch: starting from 'runs/detect/train4/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.2 MB)

ONNX: starting export with onnx 1.17.0 opset 18...


W0721 15:40:35.917000 1319 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0721 15:40:35.918000 1319 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0721 15:40:35.919000 1319 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0721 15:40:35.919000 1319 site-packages/torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_

ONNX: slimming with onnxslim 0.1.48...
ONNX: export success ✅ 2.3s, saved as 'runs/detect/train4/weights/best.onnx' (10.0 MB)

Export complete (2.5s)
Results saved to /workspace/notebooks/runs/detect/train4/weights
Predict:         yolo predict task=detect model=runs/detect/train4/weights/best.onnx imgsz=640  
Validate:        yolo val task=detect model=runs/detect/train4/weights/best.onnx imgsz=640 data=/workspace/notebooks/parasites.yaml  
Visualize:       https://netron.app
Validated ONNX model: runs/detect/train4/weights/best.onnx


Lastly, we can stress what we have learned, implement a method to run inference for all splits (train, validation, test), and save the result for visualization purposes.

In [8]:
import os
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from tqdm import tqdm

In [9]:
def run_inference_and_visualize(model_path, dataset_config, output_base_dir="output_visualizations"):
    """
    Run inference on train, validation, and test sets, draw bounding boxes, and save results
    
    Args:
        model_path: Path to trained YOLO model (.pt file)
        dataset_config: Path to dataset config file (.yaml)
        output_base_dir: Base directory to save visualization results
    """
    # Load the trained model
    model = YOLO(model_path)
    
    # Create output directories
    os.makedirs(output_base_dir, exist_ok=True)
    splits = ["train", "val", "test"]
    
    # Process each split
    for split in splits:
        print(f"Processing {split} set...")
        output_dir = os.path.join(output_base_dir, split)
        os.makedirs(output_dir, exist_ok=True)
        
        # Get image directory from config
        import yaml
        with open(dataset_config, 'r') as f:
            config = yaml.safe_load(f)
        
        dataset_root = Path(config["path"])
        image_dir = dataset_root / config[split]
        # Get all images in the directory
        image_paths = list(Path(image_dir).glob("*.jpg")) + list(Path(image_dir).glob("*.png"))
        
        # Process each image
        for img_path in tqdm(image_paths, desc=f"Visualizing {split} set"):
            # Run inference
            results = model.predict(str(img_path), conf=0.25, save=False)
            
            # Get original image
            img = cv2.imread(str(img_path))
            
            # Draw bounding boxes
            for result in results:
                boxes = result.boxes.cpu().numpy()
                
                for box in boxes:
                    # Get box coordinates
                    x1, y1, x2, y2 = box.xyxy[0].astype(int)
                    conf = box.conf[0]
                    cls = int(box.cls[0])
                    
                    # Draw bounding box
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    
                    # Draw label
                    label = f"Class {cls}: {conf:.2f}"
                    (label_width, label_height), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                    cv2.rectangle(img, (x1, y1 - label_height - 10), (x1 + label_width, y1), (0, 255, 0), -1)
                    cv2.putText(img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
            
            # Save the image with bounding boxes
            output_path = os.path.join(output_dir, img_path.name)
            cv2.imwrite(output_path, img)
            
    print(f"All visualizations saved to {output_base_dir}")


def visualize_ground_truth(dataset_config, output_base_dir="ground_truth_visualizations"):
    """
    Visualize ground truth bounding boxes from label files
    
    Args:
        dataset_config: Path to dataset config file (.yaml)
        output_base_dir: Base directory to save visualization results
    """
    # Create output directories
    os.makedirs(output_base_dir, exist_ok=True)
    splits = ["train", "val", "test"]
    
    # Load dataset config
    import yaml
    with open(dataset_config, 'r') as f:
        config = yaml.safe_load(f)
    
    # Get class names
    class_names = config.get("names", {0: "object"})
    
    # Process each split
    for split in splits:
        print(f"Processing {split} ground truth...")
        output_dir = os.path.join(output_base_dir, split)
        os.makedirs(output_dir, exist_ok=True)
        
        # Image and label directories
        dataset_root = Path(config["path"])
        image_dir = dataset_root / config[split]
        label_dir = dataset_root / "labels" / split
        
        # Get all images in the directory
        image_paths = list(Path(image_dir).glob("*.jpg")) + list(Path(image_dir).glob("*.png"))
        
        # Process each image
        for img_path in tqdm(image_paths, desc=f"Visualizing {split} ground truth"):
            # Construct path to corresponding label file
            label_path = os.path.join(label_dir, img_path.stem + ".txt")
            
            if not os.path.exists(label_path):
                print(label_path)
                print(f"Warning: No label file found for {img_path}")
                continue
                
            # Read image
            img = cv2.imread(str(img_path))
            height, width = img.shape[:2]
            
            # Read label file
            with open(label_path, 'r') as f:
                lines = f.readlines()
            
            # Draw ground truth boxes
            for line in lines:
                data = line.strip().split()
                if len(data) == 5:  # class_id, x_center, y_center, w, h
                    cls_id = int(data[0])
                    x_center = float(data[1]) * width
                    y_center = float(data[2]) * height
                    w = float(data[3]) * width
                    h = float(data[4]) * height
                    
                    # Convert to top-left and bottom-right coordinates
                    x1 = int(x_center - w/2)
                    y1 = int(y_center - h/2)
                    x2 = int(x_center + w/2)
                    y2 = int(y_center + h/2)
                    
                    # Get class name
                    class_name = class_names.get(cls_id, f"Class {cls_id}")
                    
                    # Draw bounding box (red for ground truth)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
                    
                    # Draw label
                    label = f"{class_name}"
                    (label_width, label_height), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                    cv2.rectangle(img, (x1, y1 - label_height - 10), (x1 + label_width, y1), (0, 0, 255), -1)
                    cv2.putText(img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            
            # Save the image with ground truth boxes
            output_path = os.path.join(output_dir, img_path.name)
            cv2.imwrite(output_path, img)
            
    print(f"All ground truth visualizations saved to {output_base_dir}")

In [10]:
# Example usage
model_path = Path(model.trainer.best)  # Best checkpoint from the training run
   
# Visualize model predictions
run_inference_and_visualize(model_path, dataset_config)
    
# Visualize ground truth boxes
visualize_ground_truth(dataset_config)
    
print("Visualization complete!")

Processing train set...


Visualizing train set:   0%|          | 0/731 [00:00<?, ?it/s]


image 1/1 /workspace/images/object-detection/parasites/images/train/000260.png: 640x640 1 parasite_egg, 8.5ms
Speed: 1.2ms preprocess, 8.5ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)


Visualizing train set:   0%|          | 1/731 [00:00<01:17,  9.44it/s]


image 1/1 /workspace/images/object-detection/parasites/images/train/001144.png: 640x640 (no detections), 9.6ms
Speed: 1.1ms preprocess, 9.6ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /workspace/images/object-detection/parasites/images/train/000066.png: 640x640 1 parasite_egg, 9.6ms
Speed: 1.4ms preprocess, 9.6ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /workspace/images/object-detection/parasites/images/train/000171.png: 640x640 (no detections), 5.9ms
Speed: 1.1ms preprocess, 5.9ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /workspace/images/object-detection/parasites/images/train/000967.png: 640x640 1 parasite_egg, 11.1ms
Speed: 1.4ms preprocess, 11.1ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 640)


...

/workspace/images/object-detection/parasites/labels/test/001150.txt
/workspace/images/object-detection/parasites/labels/test/000351.txt
/workspace/images/object-detection/parasites/labels/test/000492.txt


Visualizing test ground truth:  83%|████████▎ | 204/245 [00:01<00:00, 194.39it/s]

/workspace/images/object-detection/parasites/labels/test/000079.txt
/workspace/images/object-detection/parasites/labels/test/000161.txt
/workspace/images/object-detection/parasites/labels/test/000714.txt
/workspace/images/object-detection/parasites/labels/test/000480.txt
/workspace/images/object-detection/parasites/labels/test/000243.txt
/workspace/images/object-detection/parasites/labels/test/000658.txt
/workspace/images/object-detection/parasites/labels/test/000357.txt
/workspace/images/object-detection/parasites/labels/test/000868.txt
/workspace/images/object-detection/parasites/labels/test/000450.txt
/workspace/images/object-detection/parasites/labels/test/000732.txt
/workspace/images/object-detection/parasites/labels/test/000878.txt
/workspace/images/object-detection/parasites/labels/test/000474.txt
/workspace/images/object-detection/parasites/labels/test/000821.txt
/workspace/images/object-detection/parasites/labels/test/001217.txt
/workspace/images/object-detection/parasites/lab

Visualizing test ground truth: 100%|██████████| 245/245 [00:01<00:00, 203.81it/s]

/workspace/images/object-detection/parasites/labels/test/000686.txt
/workspace/images/object-detection/parasites/labels/test/000500.txt
/workspace/images/object-detection/parasites/labels/test/000299.txt
/workspace/images/object-detection/parasites/labels/test/001162.txt
/workspace/images/object-detection/parasites/labels/test/000101.txt
/workspace/images/object-detection/parasites/labels/test/000448.txt
/workspace/images/object-detection/parasites/labels/test/000530.txt
/workspace/images/object-detection/parasites/labels/test/000654.txt
/workspace/images/object-detection/parasites/labels/test/000165.txt
/workspace/images/object-detection/parasites/labels/test/001164.txt
/workspace/images/object-detection/parasites/labels/test/001020.txt
/workspace/images/object-detection/parasites/labels/test/000121.txt
/workspace/images/object-detection/parasites/labels/test/001203.txt
/workspace/images/object-detection/parasites/labels/test/000207.txt
All ground truth visualizations saved to ground_